# Taller UNAM · IA aplicada a las Finanzas
**Quantmetrics** · 5.ª Semana Financiera de la CAFE · 14 de octubre de 2026

Dos laboratorios:
1. **Riesgo de mercado:** VaR al 99 % con tres enfoques y backtesting.
2. **Riesgo de crédito:** scoring con regresión logística vs. gradient boosting, y explicabilidad.

Si no hay internet, el notebook simula datos y todo sigue funcionando.

In [ ]:
# Solo en Colab o si faltan librerías
!pip install -q arch yfinance shap scikit-learn pandas numpy scipy

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy import stats
from arch import arch_model
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

SEED = 42
rng = np.random.default_rng(SEED)
ALPHA = 0.01          # VaR al 99 %
N_TEST = 750          # días fuera de muestra

## Laboratorio 1 · Riesgo de mercado
### Paso 1. Rendimientos y volatilidad
El **rendimiento logarítmico diario** (en %) es

$$r_t = 100\,\ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

La **volatilidad** es la desviación típica de esos rendimientos: cuánto se mueven. No es constante: hay rachas tranquilas y rachas agitadas.

In [ ]:
def load_returns(ticker="^GSPC", start="2012-01-01"):
    """Rendimientos logarítmicos diarios en %. Si no hay internet, simula un GARCH-t."""
    try:
        import yfinance as yf
        px = yf.download(ticker, start=start, progress=False, auto_adjust=True)["Close"].squeeze()
        r = (100 * np.log(px).diff().dropna()).rename("r")
        if len(r) < 2000: raise ValueError("pocos datos")
        return r
    except Exception:
        n, w, a, b = 3000, 0.02, 0.09, 0.89
        r, s2 = np.zeros(n), w / (1 - a - b)
        for t in range(n):
            r[t] = np.sqrt(s2) * rng.standard_t(6) / np.sqrt(6 / 4)
            s2 = w + a * r[t] ** 2 + b * s2
        return pd.Series(r, index=pd.bdate_range("2012-01-01", periods=n), name="r")

r = load_returns()
r.plot(figsize=(10, 3), title='Rendimientos diarios (%)');
print(f'{len(r)} observaciones | desv. típica = {r.std():.2f} %')

### Paso 2. Qué es el VaR
Con confianza del 99 %, el **VaR** es la pérdida que no debería superarse en un día:

$$\mathrm{VaR}^{99\%}_{t+1} = -\bigl(\mu + \sigma_{t+1}\,q_{0{,}01}\bigr)$$

donde $q_{0{,}01}$ es el cuantil 1 % de la distribución de los rendimientos estandarizados.

**Enfoque 1 · Histórico:** el cuantil 1 % de los últimos 250 días.

In [ ]:
def t_quantile(nu, alpha=ALPHA):
    """Cuantil t estandarizado (varianza 1)."""
    return stats.t.ppf(alpha, nu) * np.sqrt((nu - 2) / nu)

def var_hist(r, window=250):
    q = r.rolling(window).quantile(ALPHA).shift(1)
    return (-q).iloc[-N_TEST:]

**Enfoque 2 · GARCH(1,1) con t-Student:** modela que la volatilidad llega en rachas,

$$\sigma^2_{t+1} = \omega + \alpha\,\varepsilon_t^2 + \beta\,\sigma_t^2$$

Se reestima cada 20 días y, entre medio, se actualiza la volatilidad con los datos nuevos.

In [ ]:
def var_garch(r, window=1000, refit=20):
    """VaR 1 día con GARCH(1,1)-t; reestima cada `refit` días y filtra entre medio."""
    out, params = {}, None
    for k, i in enumerate(range(len(r) - N_TEST, len(r))):
        m = arch_model(r.iloc[i - window:i], vol="GARCH", p=1, q=1, dist="t")
        if k % refit == 0: params = m.fit(disp="off").params
        s = np.sqrt(m.fix(params).forecast(horizon=1).variance.values[-1, 0])
        out[r.index[i]] = -(params["mu"] + s * t_quantile(params["nu"]))
    return pd.Series(out)

**Enfoque 3 · Random Forest:** muchos árboles de decisión *votan* la volatilidad de mañana usando rezagos de $|r|$ y desviaciones móviles. Todas las variables van **rezagadas** para no usar información del futuro. Luego $\sigma \approx E|r|\sqrt{\pi/2}$.

In [ ]:
def features(r):
    a = r.abs()
    X = pd.DataFrame({f"abs_l{k}": a.shift(k) for k in range(1, 6)})
    X["std5"], X["std22"], X["mabs22"] = r.rolling(5).std(), r.rolling(22).std(), a.rolling(22).mean()
    X["ret_l1"] = r.shift(1)
    X[["std5", "std22", "mabs22"]] = X[["std5", "std22", "mabs22"]].shift(1)   # sin look-ahead
    return X

def var_rf(r, window=1000, refit=60, nu=5):
    """Random Forest predice |r_t|; sigma ≈ E|r|·√(π/2); VaR con cuantil t(5)."""
    X, y = features(r), r.abs()
    out, rf = {}, None
    for k, i in enumerate(range(len(r) - N_TEST, len(r))):
        if k % refit == 0:
            tr = slice(i - window, i)
            rf = RandomForestRegressor(200, min_samples_leaf=10, n_jobs=-1, random_state=SEED)
            rf.fit(X.iloc[tr].dropna(), y.iloc[tr][X.iloc[tr].notna().all(axis=1)])
        s = rf.predict(X.iloc[[i]])[0] * np.sqrt(np.pi / 2)
        out[r.index[i]] = -(s * t_quantile(nu))
    return pd.Series(out)

### Paso 3. Backtesting con la prueba de Kupiec
Un VaR al 99 % debe fallar el 1 % de los días. Con $n$ días y $x$ violaciones, $\hat p = x/n$ y

$$LR_{POF} = -2\ln\frac{(1-p)^{n-x}\,p^{x}}{(1-\hat p)^{n-x}\,\hat p^{x}} \;\sim\; \chi^2_1,\qquad p = 0{,}01$$

Si el **p-valor > 0,05**, no rechazamos que la tasa de violaciones sea 1 %.

In [ ]:
def kupiec(x, n, p):
    """Prueba POF de Kupiec: H0 = la tasa de violaciones es p."""
    ph = x / n
    ll = lambda q: (n - x) * np.log(1 - q) + (x * np.log(q) if x > 0 else 0)
    lr = -2 * (ll(p) - ll(ph))
    return lr, 1 - stats.chi2.cdf(lr, 1)

def lab1():
    r = load_returns()
    real = r.iloc[-N_TEST:]
    res = {"Histórico (250d)": var_hist(r), "GARCH(1,1)-t": var_garch(r), "Random Forest": var_rf(r)}
    rows = []
    for name, v in res.items():
        x = int((real.values < -v.reindex(real.index).values).sum())
        lr, p = kupiec(x, len(real), ALPHA)
        rows.append({"Modelo": name, "VaR medio %": v.mean(), "Violaciones": x,
                     "Esperadas": ALPHA * len(real), "Kupiec p-valor": p,
                     "Veredicto": "Aceptado" if p > 0.05 else "Rechazado"})
    print("\n=== LAB 1 · VaR 99 % a 1 día, backtesting fuera de muestra ===")
    print(pd.DataFrame(rows).round(3).to_string(index=False))

lab1()

**Para discutir:** ¿qué modelo tiene más violaciones de las esperadas? ¿El VaR medio más bajo es mejor o solo más arriesgado?

---
## Laboratorio 2 · Riesgo de crédito
Un **scoring** estima la probabilidad de que un cliente no pague (*default*). Se compara con:
- **AUC:** probabilidad de que el modelo ponga mayor riesgo a un moroso que a un buen pagador (0,5 = azar; 1 = perfecto).
- **KS:** separación máxima entre buenos y malos pagadores (0 a 1).

In [ ]:
def make_credit(n=20000):
    d = pd.DataFrame({
        "ingreso": rng.lognormal(7.5, 0.5, n), "edad": rng.integers(21, 70, n),
        "deuda_ingreso": rng.beta(2, 5, n) * 1.2, "antiguedad": rng.exponential(5, n),
        "moras_12m": rng.poisson(0.4, n), "uso_linea": rng.beta(2, 3, n)})
    z = (-4.3 + 1.6 * d.deuda_ingreso + 0.8 * d.uso_linea + 0.25 * d.moras_12m + 1.0 * (d.moras_12m >= 2)
         - 0.03 * d.antiguedad - 0.6 * np.log(d.ingreso / 1800) + 4.0 * d.deuda_ingreso * d.uso_linea
         + 0.9 * ((d.edad < 27) | (d.edad > 62)))      # efectos no lineales: aquí gana el ML
    d["default"] = rng.binomial(1, 1 / (1 + np.exp(-z)))
    return d

d = make_credit()
print(f'Tasa de default: {d.default.mean():.1%}')
d.head()

### Modelos y métricas
La **regresión logística** es la referencia (lineal). El **gradient boosting** suma árboles pequeños que corrigen los errores del anterior y capta **no linealidades e interacciones**.

In [ ]:
def ks(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return (tpr - fpr).max()

def lab2():
    d = make_credit()
    X, y = d.drop(columns="default"), d["default"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
    lg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
    gb = GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                    subsample=0.8, random_state=SEED).fit(Xtr, ytr)
    print(f"\n=== LAB 2 · Scoring de crédito (tasa de default {y.mean():.1%}) ===")
    for name, m in [("Regresión logística", lg), ("Gradient boosting", gb)]:
        p = m.predict_proba(Xte)[:, 1]
        print(f"{name:20s} AUC = {roc_auc_score(yte, p):.3f} | KS = {ks(yte, p):.3f}")
    try:
        import shap
        sv = shap.TreeExplainer(gb).shap_values(Xte.iloc[:2000])
        imp = pd.Series(np.abs(sv).mean(0), index=X.columns)
        how = "SHAP |valor| medio"
    except Exception:
        pi = permutation_importance(gb, Xte, yte, scoring="roc_auc", n_repeats=5, random_state=SEED)
        imp, how = pd.Series(pi.importances_mean, index=X.columns), "importancia por permutación"
    print(f"\nVariables que más pesan en la decisión ({how}):")
    print(imp.sort_values(ascending=False).round(4).to_string())

lab2()

### Explicabilidad
**SHAP** reparte la predicción entre las variables: cuánto empuja cada una hacia más o menos riesgo. Si SHAP no está instalado, se usa la **importancia por permutación**.

> Un modelo que no se puede explicar no se puede aprobar ante el regulador.

**Cierre:** la IA complementa al modelo clásico, necesita validación y explicabilidad, y el valor está en la decisión de riesgo.